<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m4_02_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M4.02 — Teaching a straight line to follow the clock

In the lab we made linear models fit curved data by **transforming the
features**: polynomials, bins, splines. In this workbook you repair the
model of workbook M4.01, which could not see that traffic rises in the
morning and falls at night. You will compare four ways of describing the
hour of the day, let workdays and days off have their own rhythm, and
see what a polynomial does when it is asked about the future.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "better_description": "0e353d7a66a91a3b",
    "better_on_future": "6b86b273ff34fce1",
    "better_on_past": "2c624232cdd22177",
    "interaction_gain": "cc421e18139f05a3",
    "n_columns_with_interaction": "0e17daca5f3e175f",
    "n_onehot_columns": "c2356069e9d1e79c"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data, and a test set in the future

The same motorway as in workbook M4.01: one row per hour of 2016 and
2017, and `traffic_volume` (vehicles per hour) to predict. This time we
only use the **calendar**:

- `hour`: the hour of the day, 0 to 23;
- `workday`: 1 from Monday to Friday unless it is a public holiday, 0
  otherwise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

traffic = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/metro_traffic.csv",
    parse_dates=["date_time"],
)
traffic["hour"] = traffic["date_time"].dt.hour
traffic["workday"] = (
    (traffic["date_time"].dt.day_of_week < 5) & (traffic["holiday"] == "No holiday")
).astype(int)
traffic[["date_time", "hour", "workday", "traffic_volume"]].head()

The model will be used to predict traffic **in the coming months**. So we
test it the same way: it learns from the first 18 months and is tested
on the last 6. A shuffled split would let it learn from hours that come
*after* the ones it is tested on.

In [ ]:
from sklearn.metrics import mean_absolute_error

train = traffic[traffic["date_time"] < "2017-07-01"]
test = traffic[traffic["date_time"] >= "2017-07-01"]
baseline_error = mean_absolute_error(
    test["traffic_volume"], np.full(len(test), train["traffic_volume"].mean())
)
print(f"{len(train)} hours to learn from, {len(test)} hours to test on")
print(f"always predicting the mean: error {baseline_error:.0f} vehicles per hour")

All the models below are the **same linear model** (`Ridge`, a linear
regression with a small safety margin that workbook M4.03 explains).
Only the description of the features changes. Two helpers do the
repetitive work: one trains and reports the test error, the other draws
four days of the test period, from a Thursday to a Sunday.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

fitted_models, errors = {}, {}


def evaluate(name, preprocessor):
    """Train preprocessor + linear model, store it, return its test error."""
    model = make_pipeline(preprocessor, Ridge())
    model.fit(train[["hour", "workday"]], train["traffic_volume"])
    fitted_models[name] = model
    errors[name] = mean_absolute_error(
        test["traffic_volume"], model.predict(test[["hour", "workday"]])
    )
    print(f"{name}: error {errors[name]:.0f} vehicles per hour")


def plot_four_days(names):
    days = test[(test["date_time"] >= "2017-09-07") & (test["date_time"] < "2017-09-11")]
    fig, axes = plt.subplots(len(names), 1, figsize=(11, 2.6 * len(names)), sharex=True, squeeze=False)
    for ax, name in zip(axes[:, 0], names):
        ax.plot(days["date_time"], days["traffic_volume"], color="lightgray", linewidth=4, label="real traffic")
        ax.plot(days["date_time"], fitted_models[name].predict(days[["hour", "workday"]]), color="black", label="prediction")
        ax.set(title=f"{name} (error {errors[name]:.0f})", ylabel="vehicles per hour")
    axes[0, 0].legend(loc="upper right", fontsize=8)
    axes[-1, 0].set_xlabel("Thursday 7 to Sunday 10 September 2017")

## 2. The hour as a number

First the naive version: the hour is passed as it is, a number from 0 to
23, simply scaled.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler

evaluate(
    "hour as a number",
    make_column_transformer((StandardScaler(), ["hour"]), ("passthrough", ["workday"])),
)
plot_four_days(["hour as a number"])

The prediction is a ramp that climbs all day and falls back at midnight,
as in workbook M4.01. The model multiplies the hour by one weight and
can do nothing else.

## 3. Three better descriptions of the hour

**One-hot encoding** treats the hour as a category, as in module 1: one
column per hour of the day, and one weight for each.

✏️ How many columns does one-hot encoding create from `hour`?

In [ ]:
n_onehot_columns = ____
check("n_onehot_columns", n_onehot_columns)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot_hour = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
evaluate(
    "hour one-hot",
    make_column_transformer((one_hot_hour, ["hour"]), ("passthrough", ["workday"])),
)

The error is divided by more than two. But one-hot encoding has
forgotten something we know: hours follow each other, and **23:00 is
next to 00:00**. For the model, the 24 hours are 24 unrelated things.

A classic trick puts the hour on a **clock face**: two columns, the sine
and the cosine of the angle of the hour hand. 23:00 and 00:00 are then
close to each other, as they should be.

🔮 **Predict first.** Will the sine/cosine description give a **"lower"**,
**"about equal"** or **"higher"** error than one-hot encoding?

In [ ]:
my_prediction = ____
commit(my_prediction, ["lower", "about equal", "higher"])

In [ ]:
from sklearn.preprocessing import FunctionTransformer


def clock_face(hours):
    angle = 2 * np.pi * np.asarray(hours) / 24
    return np.hstack([np.sin(angle), np.cos(angle)])


evaluate(
    "hour as sine and cosine",
    make_column_transformer((FunctionTransformer(clock_face), ["hour"]), ("passthrough", ["workday"])),
)
plot_four_days(["hour one-hot", "hour as sine and cosine"])

✏️ Which description gives the lower error? Write `"one-hot"` or
`"sine and cosine"`.

In [ ]:
better_description = ____
check("better_description", better_description)

The elegant trick loses. With two columns and two weights, the model can
only draw **one smooth wave per day**. Real traffic has a morning peak,
a midday plateau and an evening peak: too many bumps for one wave.
One-hot encoding has 24 weights and can follow any daily shape.

**Periodic splines** take the best of both. A spline describes the hour
with a few smooth bumps placed around the clock, each with its own
weight, and the last bump joins the first one across midnight.

In [ ]:
from sklearn.preprocessing import SplineTransformer

periodic_spline = SplineTransformer(
    degree=3,
    knots=np.linspace(0, 24, 13).reshape(-1, 1),
    extrapolation="periodic",
)
evaluate(
    "hour as periodic spline",
    make_column_transformer((periodic_spline, ["hour"]), ("passthrough", ["workday"])),
)

hours = pd.DataFrame({"hour": np.linspace(0, 24, 200)})
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(hours["hour"], periodic_spline.fit(train[["hour"]]).transform(hours))
_ = ax.set(title="The 12 columns created by the periodic spline", xlabel="hour of the day", xticks=range(0, 25, 4))

Each curve is one new column. With 12 columns, half as many as one-hot
encoding, the spline reaches practically the same error, and its
prediction is a smooth curve through the day.

In [ ]:
pd.Series(errors).round(0).to_frame("test error")

scikit-learn's example
[Time-related feature engineering](https://scikit-learn.org/1.8/auto_examples/applications/plot_cyclical_feature_engineering.html)
runs this comparison on bike rentals, with the same surprise for the
sine and cosine.

🧠 **Think.** The model with the hour as a number has 1 weight for the
hour, the sine/cosine model 2, the spline 12 and one-hot 24. Explain the
four errors with the vocabulary of module 2: which models underfit?

## 4. ➕ Going further: workdays do not look like Sundays

Look again at the last plot of four days. On Saturday and Sunday the
prediction has the **same shape** as on Thursday and Friday, only
shifted down. Our models are **additive**:

`prediction = effect of the hour + effect of the workday`.

The workday can move the whole curve up or down and nothing more. Real
Sundays have no morning rush at all.

An **interaction** gives the model what it lacks: new columns that are
the product of an hour column and the workday column. Such a column is
"it is 8:00 **and** it is a workday", with its own weight.
`PolynomialFeatures(interaction_only=True)` adds all the products of two
columns.

✏️ We start from the 24 one-hot hour columns and the `workday` column,
and add the products of `workday` with each hour column. (The products
of two different hour columns are always zero and carry nothing.) How
many **useful** columns do we have in total?

In [ ]:
n_columns_with_interaction = ____
check("n_columns_with_interaction", n_columns_with_interaction, hint="The original columns are kept, and one product is added per hour.")

🔮 **Predict first.** Will the interaction reduce the error **"not at
all"**, **"a little"** (less than 10%) or **"a lot"** (more than 30%)?

In [ ]:
my_prediction = ____
commit(my_prediction, ["not at all", "a little", "a lot"])

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

evaluate(
    "hour one-hot × workday",
    make_pipeline(
        make_column_transformer(
            (OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["hour"]),
            ("passthrough", ["workday"]),
        ),
        PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
    ),
)
plot_four_days(["hour one-hot", "hour one-hot × workday"])

✏️ How much did the interaction reduce the error? Write `"not at all"`,
`"a little"` or `"a lot"`.

In [ ]:
interaction_gain = ____
check("interaction_gain", interaction_gain)

Workdays and days off now have their own daily curve. We added **no new
information**: the model had the hour and the workday all along. What it
lacked was the permission to combine them.

For comparison, a decision tree given the same two raw columns, with no
feature engineering at all:

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(random_state=0).fit(train[["hour", "workday"]], train["traffic_volume"])
tree_error = mean_absolute_error(test["traffic_volume"], tree.predict(test[["hour", "workday"]]))
print(f"decision tree on the raw hour and workday: error {tree_error:.0f} vehicles per hour")
print(f"linear model with one-hot × workday:       error {errors['hour one-hot × workday']:.0f} vehicles per hour")

🧠 **Think.** The tree reaches the same error without any help. From
what you know about how a tree splits the data, why does it not need
one-hot encoding or interactions? What does the linear model offer in
exchange for the extra work?

## 5. ➕ Going further: a polynomial asked about the future

Traffic also changes slowly over the months. We compute the **mean
traffic of each week** and try to describe its evolution with a curve,
as a function of the week number. The model learns from the first 18
months and we then ask it about the 6 following months, which lie
**outside** the range of week numbers it has seen.

In [ ]:
weekly = (
    traffic.set_index("date_time")["traffic_volume"].resample("W").mean().dropna().reset_index()
)
weekly["week"] = np.arange(len(weekly))
past = weekly[weekly["date_time"] < "2017-07-01"]
future = weekly[weekly["date_time"] >= "2017-07-01"]
print(f"{len(past)} weeks to learn from, {len(future)} weeks to predict")

🔮 **Predict first.** We fit a straight line (degree 1) and a polynomial
of degree 8. The degree-8 curve follows the past weeks more closely.
Which one will predict the **future** weeks better? `"degree 1"` or
`"degree 8"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["degree 1", "degree 8"])

In [ ]:
from sklearn.linear_model import LinearRegression

trend_errors, trend_predictions = {}, {}
for degree in [1, 8]:
    trend = make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())
    trend.fit(past[["week"]], past["traffic_volume"])
    trend_predictions[degree] = trend.predict(weekly[["week"]])
    trend_errors[degree] = {
        "error on the past": mean_absolute_error(past["traffic_volume"], trend.predict(past[["week"]])),
        "error on the future": mean_absolute_error(future["traffic_volume"], trend.predict(future[["week"]])),
    }

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(weekly["date_time"], weekly["traffic_volume"], color="lightgray", linewidth=4, label="real weekly mean")
for degree, predicted in trend_predictions.items():
    ax.plot(weekly["date_time"], predicted, label=f"degree {degree}")
ax.axvline(pd.Timestamp("2017-07-01"), linestyle="--", color="red")
ax.text(pd.Timestamp("2017-07-08"), 500, "the future", color="red")
ax.set(ylim=(0, 6000), ylabel="vehicles per hour (weekly mean)")
ax.legend(loc="lower left")
pd.DataFrame(trend_errors).T.rename_axis("degree").round(0)

✏️ Which degree is better **on the past**, and which is better **on the
future**? Enter `1` or `8` for each.

In [ ]:
better_on_past = ____
better_on_future = ____
check("better_on_past", better_on_past)
check("better_on_future", better_on_future)

Inside the range it was trained on, a high-degree polynomial is a
flexible curve. Outside, it is driven by its highest power and leaves
for the sky or the floor. The terms `week⁸` do not know that traffic
cannot be negative.

**Extrapolation**, predicting outside the range of the training data, is
the weak point of every flexible feature transformation. A test set
shuffled among the training weeks would never have shown it: this is why
section 1 put the test set in the future.

🧠 **Think.**

1. The `hour` feature never causes this problem: why? (Which hours will
   the model meet next year?)
2. The straight line ignores the dips around each new year. What
   information, absent from the week number, would describe them?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the four descriptions of the hour and why sine/cosine lost against
     one-hot;
   - what an interaction is and what it changed for Saturdays and
     Sundays;
   - the tree against the engineered linear model;
   - the polynomial of degree 8 and the future: what went wrong and how
     the test set was chosen to reveal it.